# Exp4: 基于K-近邻的车牌号识别

## 一、案例简介

图像的智能处理一直是人工智能领域广受关注的一类技术，代表性的如人脸识别与 CT 肿瘤识别，在人工智能落地的进程中发挥着重要作用。其中车牌号识别作为一个早期应用场景，已经融入日常生活中，为我们提供了诸多便利，在各地的停车场和出入口都能看到它的身影。车牌号识别往往分为字符划分和字符识别两个子任务，本案例我们将关注字符识别的任务，尝试用 K-NN 的方法对分割好的字符图像进行自动识别和转化。

## 二、作业说明

### 基本要求
* 完成数据的读入和表示，将图片表示成向量并和 label 对应上；
* 构建 K-NN 模型（可调库）对测试集中的图片进行预测并计算准确率；
* 分析当 K 取不同值时测试准确率的变化。

### 扩展要求
* 分析不同距离度量方式对模型效果的影响；
* 对比平权和加权 K-NN 的效果；
* 分析训练集大小对测试结果的影响。

## 三、数据概览
本次我们使用已经分割好的车牌图片作为数据集，包括数字 0-9、字母 A-Z（不包含 O 和 I）以及省份简称共 65 个类，编号从 0 到 64。数据已经分成了训练集和测试集，里面的文件夹用 label 编号命名，一个文件夹下的所有图片都属于该文件夹对应的类，每个图片都是 20 * 20 的二值化灰度图。

## 四、实验报告

### 导入相关库

In [ ]:
%matplotlib inline
import os  # 导入操作系统相关功能，用于文件和目录操作
import numpy as np  # 导入NumPy库，用于高性能科学计算和多维数组处理
import seaborn as sns  # 导入Seaborn库，用于创建美观的统计图表
import matplotlib.pyplot as plt  # 导入Matplotlib的pyplot模块，用于创建各种可视化图表
from PIL import Image  # 从PIL库导入Image模块，用于图像处理和操作
from typing import (
    Tuple,
    Dict,
    Optional,
    List,
)  # 从typing模块导入类型提示相关功能，增强代码的可读性和可维护性
from sklearn.neighbors import (
    KNeighborsClassifier,
)  # 从sklearn库导入K近邻分类器，用于实现KNN机器学习算法
from sklearn.model_selection import (
    train_test_split,
)  # 从sklearn库导入训练集和测试集分割功能
from sklearn.metrics import (
    accuracy_score,
    classification_report,
)  # 从sklearn库导入评估指标，用于模型性能评估

### 环境配置

In [ ]:
plt.rcParams["font.family"] = [
    "SimHei"
]  # 配置Matplotlib使用中文字体，确保中文可以正常显示
plt.rcParams["axes.unicode_minus"] = False  # 配置Matplotlib正确显示负号
plt.rcParams["font.size"] = 12  # 设置Matplotlib的字体大小
sns.set_theme(style="whitegrid", font="SimHei")  # 设置Seaborn的绘图风格为白色网格风格
RANDOM_SEED = 2025

### 实验类的定义

In [ ]:
class LicensePlateKNN:
    def __init__(self) -> None:
        """
        初始化车牌字符识别KNN模型
        """

        self.X_train: Optional[np.ndarray] = None
        self.y_train: Optional[np.ndarray] = None
        self.X_test: Optional[np.ndarray] = None
        self.y_test: Optional[np.ndarray] = None
        self.model: Optional[KNeighborsClassifier] = None
        self.best_params: Dict[str, any] = {
            "n_neighbors": 3,
            "metric": "euclidean",
            "weights": "uniform",
        }

    def _load_images(self, data_dir: str) -> Tuple[np.ndarray, np.ndarray]:
        """
        加载图像数据并预处理

        :param data_dir: 数据目录路径
        :return: 图像数据数组和标签数组
        """

        images, labels, numeric_labels = [], [], []

        for label in os.listdir(data_dir):
            try:
                numeric_labels.append(int(label))
            except ValueError:  # 忽略非数字标签
                pass  # print(f"跳过非数字标签: {label}")

        for label in sorted(numeric_labels):
            label_dir = os.path.join(data_dir, str(label))
            if not os.path.isdir(label_dir):
                continue

            for img_file in os.listdir(label_dir):
                try:
                    img_path = os.path.join(label_dir, img_file)
                    with Image.open(img_path) as img:
                        img = img.convert("L").resize(
                            (20, 20)
                        )  # 将图像转化为灰度图，并调整为统一尺寸
                        pixels = np.array(
                            img
                        ).flatten()  # 将图像转化为Numpy数组，并展平为一维数组
                        images.append(pixels)
                        labels.append(label)
                except Exception as e:
                    print(f"加载{img_path}失败: {e}")

        return np.array(images), np.array(labels)

    def load_datasets(
        self, train_dir: str = "data/train", test_dir: str = "data/test"
    ) -> "LicensePlateKNN":
        """
        加载训练和测试数据集

        :param train_dir: 训练数据目录，默认为'data/train'
        :param test_dir: 测试数据目录，默认为'data/test'
        :return: 模型实例本身
        """

        print("加载训练数据...")
        self.X_train, self.y_train = self._load_images(train_dir)
        print(f"成功加载 {len(self.X_train)} 个训练样本")

        print("加载测试数据...")
        self.X_test, self.y_test = self._load_images(test_dir)
        print(f"成功加载 {len(self.X_test)} 个测试样本")

        if len(self.X_train) == 0:
            raise ValueError("训练数据为空，请检查数据路径")
        if len(self.X_test) == 0:
            raise ValueError("测试数据为空，请检查数据路径")

        return self

    def train_model(
        self,
        n_neighbors: Optional[int] = None,
        weights: Optional[str] = None,
        metric: Optional[str] = None,
    ) -> "LicensePlateKNN":
        """
        训练KNN模型进行车牌识别

        :param n_neighbors: 最近邻数量
        :param weights: 权重计算方式，默认为'distance'
        :param metric: 距离度量方式，默认为'euclidean'
        :return: 训练完成的LincensePlateKNN实例
        """

        # 验证参数合法性
        if n_neighbors is not None and (
            n_neighbors <= 0 or not isinstance(n_neighbors, int)
        ):
            raise ValueError("n_neighbors必须是正整数")
        if weights is not None and weights not in ["uniform", "distance"]:
            raise ValueError("权重方式必须是'uniform'或'distance'")
        if metric is not None and metric not in [
            "manhattan",
            "euclidean",
            "chebyshev",
            "cosine",
        ]:
            raise ValueError(
                "不支持的距离度量，请选择: {['manhattan', 'euclidean', 'chebyshev', 'cosine']}"
            )

        params = {  # 合并参数
            "n_neighbors": n_neighbors or self.best_params.get("n_neighbors", 5),
            "weights": weights or self.best_params.get("weights", "distance"),
            "metric": metric or self.best_params.get("metric", "euclidean"),
        }

        self.model = KNeighborsClassifier(**params)
        self.model.fit(self.X_train, self.y_train)
        print(
            f"模型训练完成.\n参数: n_neighbors = {params['n_neighbors']}, ",
            f"metric = {params['metric']}, ",
            f"weights = {params['weights']}.",
        )

        return self

    def evaluate(self, detailed: bool = False) -> Tuple[float, float]:
        """
        评估模型性能，支持详细评估报告

        :param detailed: 是否输出详细分类报告
        :return: 训练集和测试集准确率
        """

        if self.model is None:
            raise ValueError("模型未训练，请先训练模型.")

        train_pred = self.model.predict(self.X_train)
        test_pred = self.model.predict(self.X_test)

        train_acc = accuracy_score(self.y_train, train_pred)
        test_acc = accuracy_score(self.y_test, test_pred)

        if detailed:  # 输出详细分类报告
            print("\n训练集分类报告:")
            print(classification_report(self.y_train, train_pred))
            print("\n测试集分类报告:")
            print(classification_report(self.y_test, test_pred))

        return train_acc, test_acc

    def predict(self, image: np.ndarray) -> int:
        """
        对单张图像进行预测

        :param image: 预处理后的图像数据
        :return: 预测的标签
        """

        if self.model is None:
            raise ValueError("模型未训练，请先训练模型.")

        if image.ndim == 1:  # 确保输入是二维数组
            image = image.reshape(1, -1)

        return self.model.predict(image)[0]

    def predict_and_visualize(self, image_path: str) -> None:
        """
        预测图像标签并可视化结果

        :param image_path: 图像文件路径
        """

        try:
            with Image.open(image_path) as img:  # 加载并预处理图像
                img = img.convert("L").resize((20, 20))
                pixels = np.array(img).flatten()
                prediction = self.predict(pixels)  # 预测标签
                plt.figure(figsize=(4, 4))  # 显示图像和预测结果
                plt.imshow(img, cmap="gray")
                plt.title(f"预测结果: {prediction}", color="purple")
                plt.axis("off")
                plt.show()
                # print(f"图像 {image_path} 预测标签: {prediction}")

        except Exception as e:
            print(f"预测图像 {image_path} 失败: {e}")

    def _optimize_k(self, k_range: range, visualize: bool = True) -> None:
        """
        优化K值参数

        :param k_range: K值搜索范围，默认为[1,20]
        :param visualize: 是否显示优化过程图标，默认为True
        """

        train_accs, test_accs = [], []

        for k in k_range:
            self.train_model(n_neighbors=k)
            train_acc, test_acc = self.evaluate()
            train_accs.append(train_acc)
            test_accs.append(test_acc)
            print(f"训练集准确率 = {train_acc:.2%}, 测试集准确率 = {test_acc:.2%}.\n")

        best_k = k_range[np.argmax(test_accs)]
        self.best_params["n_neighbors"] = best_k

        if visualize:
            plt.figure(figsize=(14, 6))
            x = np.arange(1, len(k_range) + 1)

            plt.subplot(1, 2, 1)
            plt.plot(k_range, train_accs, "o-", label="训练集准确率")
            plt.plot(k_range, test_accs, "s-", label="测试集准确率")
            plt.title("K值优化分析", color="purple")
            plt.xlabel("K值", color="blue")
            plt.ylabel("准确率", color="green")
            plt.ylim(0, 1.1)
            plt.legend(loc="lower left")
            plt.xticks(x, k_range)
            plt.grid(True)

            plt.subplot(1, 2, 2)
            bars = plt.bar(x, test_accs)
            plt.title(
                f"最佳K值 = {best_k} (测试集准确率 = {test_accs[np.argmax(test_accs)]:.2%})",
                color="purple",
            )
            plt.xlabel("K值", color="blue")
            plt.ylabel("测试集准确率", color="green")
            plt.xticks(x, k_range)
            plt.grid(axis="y")
            # plt.axvline(x = list(k_range).index(best_k), color = 'r', linestyle = '--', label = f'最佳K值: {best_k}')

            plt.tight_layout()
            plt.show()

    def _optimize_metric(self, visualize: bool = True, metrics: list = None) -> None:
        """
        优化距离度量参数

        :param visualize: 是否显示优化过程图标，默认为True
        :param metrics: 可选的距离度量方式
        """

        if metrics == None:
            metrics = ["manhattan", "euclidean", "chebyshev", "cosine"]

        train_results, test_results = {}, {}
        unsupported = []

        for metric in metrics:
            try:
                self.train_model(metric=metric)
                train_acc, test_acc = self.evaluate()
                train_results[metric] = train_acc
                test_results[metric] = test_acc
                print(
                    f"训练集准确率 = {train_acc:.2%}, 测试集准确率 = {test_acc:.2%}.\n"
                )
            except Exception as e:
                unsupported.append(metric)
                print(f"不支持的度量 {metric}: {e}")

        if test_results:
            best_metric = max(test_results, key=test_results.get)
            self.best_params["metric"] = best_metric
            print(
                f"最佳度量: {best_metric}, 测试集准确率 = {test_results[best_metric]:.2%}.\n"
            )

            if visualize:
                valid_metrics = [m for m in metrics if m in test_results]
                plt.figure(figsize=(12, 6))
                x = np.arange(len(valid_metrics))
                width = 0.35

                train_bars = plt.bar(
                    x - width / 2,
                    [train_results[m] for m in valid_metrics],
                    width,
                    label="训练集准确率",
                )
                test_bars = plt.bar(
                    x + width / 2,
                    [test_results[m] for m in valid_metrics],
                    width,
                    label="测试集准确率",
                )
                # plt.axvline(x = valid_metrics.index(best_metric), color = 'r', linestyle = '--', label = f'最佳: {best_metric}')

                for bar in train_bars:
                    height = bar.get_height()
                    plt.text(
                        bar.get_x() + bar.get_width() / 2,
                        height + 0.02,
                        f"{height:.2%}",
                        ha="center",
                        va="bottom",
                        fontsize=12,
                    )

                for bar in test_bars:
                    height = bar.get_height()
                    plt.text(
                        bar.get_x() + bar.get_width() / 2,
                        height + 0.02,
                        f"{height:.2%}",
                        ha="center",
                        va="bottom",
                        fontsize=12,
                    )

                plt.title(f"距离度量对比 (最佳 = {best_metric})", color="purple")
                plt.xlabel("距离度量", color="blue")
                plt.ylabel("准确率", color="green")
                plt.ylim(0, 1.1)
                plt.xticks(x, valid_metrics, fontsize=14)
                plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
                plt.grid(axis="y")
                plt.tight_layout()
                plt.show()
        else:
            print("没有找到有效度量")

    def _optimize_weights(self, visualize: bool = True, weights: list = None) -> None:
        """
        优化权重方式参数

        :param visualize: 是否显示优化过程图标，默认为True
        :param weights: 可选的权重方式
        """

        if weights == None:
            weights = ["uniform", "distance"]

        train_results, test_results = {}, {}

        for weight in weights:
            self.train_model(weights=weight)
            train_acc, test_acc = self.evaluate()
            train_results[weight] = train_acc
            test_results[weight] = test_acc
            print(f"训练集准确率 = {train_acc:.2%}, 测试集准确率 = {test_acc:.2%}.\n")

        best_weight = max(test_results, key=test_results.get)
        self.best_params["weights"] = best_weight
        print(
            f"最佳权重: {best_weight}, 测试集准确率 = {test_results[best_weight]:.2%}\n"
        )

        if visualize:
            plt.figure(figsize=(12, 6))
            x = np.arange(len(weights))
            width = 0.35

            train_bars = plt.bar(
                x - width / 2, train_results.values(), width, label="训练集准确率"
            )
            test_bars = plt.bar(
                x + width / 2, test_results.values(), width, label="测试集准确率"
            )

            for bar in train_bars:
                height = bar.get_height()
                plt.text(
                    bar.get_x() + bar.get_width() / 2,
                    height + 0.02,
                    f"{height:.2%}",
                    ha="center",
                    va="bottom",
                    fontsize=16,
                )

            for bar in test_bars:
                height = bar.get_height()
                plt.text(
                    bar.get_x() + bar.get_width() / 2,
                    height + 0.02,
                    f"{height:.2%}",
                    ha="center",
                    va="bottom",
                    fontsize=16,
                )

            plt.title(f"权重方式对比", color="purple")
            plt.xlabel("权重方式", color="blue")
            plt.ylabel("准确率", color="green")
            plt.ylim(0, 1.1)
            plt.xticks(x, weights, fontsize=18)
            plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            plt.grid(axis="y")
            plt.tight_layout()
            plt.show()

    def optimize_parameters(
        self, k_range: range = range(1, 21), visualize: bool = True
    ) -> Dict[str, any]:
        """
        优化模型参数，支持跳过可视化

        :param k_range: K值搜索范围，默认为[1,20]
        :param visualize: 是否显示优化过程图标，默认为True
        :return: 最佳参数字典
        """

        print("\n开始优化K值...")
        self._optimize_k(k_range, visualize)

        print("\n开始优化距离度量...")
        self._optimize_metric(visualize)

        print("\n开始优化权重方式...")
        self._optimize_weights(visualize)

        print(f"\n最佳参数: {self.best_params}")
        return self.best_params

    def analyze_training_size(
        self, sizes: np.ndarray = np.linspace(0.01, 0.99, 20), visualize: bool = True
    ) -> None:
        """
        分析训练集大小对准确率的影响

        :param size: 包含训练集大小所占比例的列表
        :param visualize: 是否显示优化过程图标，默认为True
        """

        train_accs, test_accs = [], []

        for size in sizes:
            if not (0.0 < size < 1.0):
                raise ValueError("训练集比例需在(0,1)之间")

            X_sub, _, y_sub, _ = train_test_split(
                self.X_train, self.y_train, train_size=size, random_state=RANDOM_SEED
            )

            temp_model = KNeighborsClassifier(**self.best_params)
            temp_model.fit(X_sub, y_sub)

            train_sub_pred = temp_model.predict(X_sub)
            test_pred = temp_model.predict(self.X_test)

            train_acc = accuracy_score(y_sub, train_sub_pred)
            test_acc = accuracy_score(self.y_test, test_pred)

            train_accs.append(train_acc)
            test_accs.append(test_acc)
            print(
                f"训练集比例={size:.2%}, 训练集准确率={train_acc:.2%}, 测试集准确率={test_acc:.2%}."
            )

        if visualize:
            plt.figure(figsize=(12, 6))
            plt.plot(sizes, train_accs, "o-", label="训练集准确率")
            plt.plot(sizes, test_accs, "s-", label="测试集准确率")
            plt.title("训练集大小对准确率的影响分析", color="purple")
            plt.xlabel("训练集比例", color="blue")
            plt.ylabel("准确率", color="green")
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.show()

    def __str__(self) -> str:
        return f"LicensePlateKNN(best_params={self.best_params})"

    def __repr__(self) -> str:
        return f"LicensePlateKNN(best_params={self.best_params})"

### 运行与结果展示(包含所有实验要求)

In [ ]:
print("初始化车牌识别模型...")
recognizer = LicensePlateKNN()
recognizer.load_datasets()

In [ ]:
print("优化模型参数...")
_ = recognizer.optimize_parameters()

In [ ]:
print("使用最佳参数训练模型...")
recognizer.train_model()

In [ ]:
print("详细评估模型...")
_ = recognizer.evaluate(detailed=True)

In [ ]:
print("分析训练集大小影响...")
recognizer.analyze_training_size()

In [ ]:
print("示例：预测单张图像.")  # 预测单张图像
recognizer.predict_and_visualize("data/test/6/1509806878_75_6.jpg")